In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/processed/cleaned.csv')
print("Shape:", df.shape)
df.head()

Shape: (2927, 78)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,...,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,...,0,0,120,0,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,...,0,0,0,0,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,Reg,Lvl,AllPub,...,0,0,0,0,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,3,2010,WD,Normal,189900


In [2]:
quality_map = {
    'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'No': 0
}

ordinal_cols = [
    'Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond',
    'Heating QC', 'Kitchen Qual', 'Fireplace Qu',
    'Garage Qual', 'Garage Cond'
]

for col in ordinal_cols:
    df[col] = df[col].map(quality_map)

print("Ordinal encoding done")
print(df[ordinal_cols].head())

Ordinal encoding done
   Exter Qual  Exter Cond  Bsmt Qual  Bsmt Cond  Heating QC  Kitchen Qual  \
0           3           3          3          4           2             3   
1           3           3          3          3           3             3   
2           3           3          3          3           3             4   
3           4           3          3          3           5             5   
4           3           3          4          3           4             3   

   Fireplace Qu  Garage Qual  Garage Cond  
0             4            3            3  
1             0            3            3  
2             0            3            3  
3             3            3            3  
4             3            3            3  


Ordinal encoding is esentially giving a numerical value to a quality. Such as a star rating system, it is not binary, and the scale differs depending on each situation

hot encoding is a binary, either a 1 or a 0. It is sort of like a yes or no question

In [3]:
df['Bsmt Exposure'] = df['Bsmt Exposure'].map(
    {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'no': 0}
)

df['Garage Finish'] = df['Garage Finish'].map(
    {'Fin': 3, 'RFn': 2, 'Unf': 1, 'No': 0}
)

df['BsmtFin Type 1'] = df['BsmtFin Type 1'].map(
    {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'No': 0}
)

df['BsmtFin Type 2'] = df['BsmtFin Type 2'].map(
    {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'No': 0}
)

In [4]:
df['SalePrice'] = np.log1p(df['SalePrice'])
print("SalePrice range after log transform:", df['SalePrice'].min(), "-", df['SalePrice'].max())

SalePrice range after log transform: 9.456418894572888 - 13.534474352733596


We do log_transform because we are essentially making the distribution more normal, so the model trains evenly accross diferent price ranges

In [5]:
# Order and PID are just identifiers, not features
# Mo Sold and Yr Sold are time-based and can cause leakage
drop_cols = ['Order', 'PID', 'Mo Sold', 'Yr Sold']
print("Shape before dropping identifier cols:", df.shape)
df.drop(columns=drop_cols, inplace=True)
print("Shape after dropping identifier cols:", df.shape)

Shape before dropping identifier cols: (2927, 78)
Shape after dropping identifier cols: (2927, 74)


We can drop the columns because we can infer the missing data and avoid redundancy. This is because of one-hot encoding. Which means if one attribute is true then the rest of attributes is false. So then in a group of attributes where this happens, we can inferr the missing detail. If they are all false, then the missing column was true, if one of the attributes is true, then the missing column was false

In [6]:
# Get remaining object columns after ordinal encoding
nominal_cols = df.select_dtypes(include='str').columns.tolist()
print("Columns to one-hot encode:", nominal_cols)

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
print("Shape after one-hot encoding:", df.shape)

Columns to one-hot encode: ['MS Zoning', 'Street', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Foundation', 'Heating', 'Central Air', 'Electrical', 'Functional', 'Garage Type', 'Paved Drive', 'Sale Type', 'Sale Condition']
Shape after one-hot encoding: (2927, 212)


In [7]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2341, 211)
X_test shape: (586, 211)
y_train shape: (2341,)
y_test shape: (586,)


In [8]:
# Only scale numeric columns — one-hot encoded columns are already 0/1
numeric_cols = X_train.select_dtypes(include='number').columns.tolist()

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling done")
print("X_train sample stats:\n", X_train[numeric_cols].describe().loc[['mean', 'std']].round(2))

Scaling done
X_train sample stats:
       MS SubClass  Lot Frontage  Lot Area  Overall Qual  Overall Cond  \
mean         -0.0          -0.0      -0.0           0.0           0.0   
std           1.0           1.0       1.0           1.0           1.0   

      Year Built  Year Remod/Add  Mas Vnr Area  Exter Qual  Exter Cond  ...  \
mean        -0.0             0.0           0.0         0.0         0.0  ...   
std          1.0             1.0           1.0         1.0         1.0  ...   

      Garage Area  Garage Qual  Garage Cond  Wood Deck SF  Open Porch SF  \
mean         -0.0         -0.0         -0.0           0.0            0.0   
std           1.0          1.0          1.0           1.0            1.0   

      Enclosed Porch  3Ssn Porch  Screen Porch  Pool Area  Misc Val  
mean            -0.0         0.0           0.0        0.0       0.0  
std              1.0         1.0           1.0        1.0       1.0  

[2 rows x 47 columns]


fit_transform is essentially calculating the standard deviation, and median based on the data given and applying it to the data.

Transforming is grabbing that previously calculated data, and applying it to the new data given. 

In [9]:
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(scaler, '../models/scaler.pkl')

print("All files saved")
print("Final feature count:", X_train.shape[1])

All files saved
Final feature count: 211
